# Local Fusion / Hybrid Retrieval for RAG

Roadmap position:

- Stage A is complete: simple RAG, chunking, evaluation, config comparison, cross-encoder reranking, and LLM-based reranking basics.
- Current topic: Stage B -> Fusion / Hybrid Retrieval.

Core production pattern:

```text
hybrid retrieval for recall
-> reranking for precision
-> generation from the best context
```


## What We Are Learning

Dense vector search and BM25 keyword search solve different retrieval problems.

Dense vector search is good when the query and document use different words but have similar meaning.

BM25 is good when exact words, names, numbers, acronyms, dates, or technical terms matter.

Fusion retrieval combines both so the initial candidate set is stronger. After that, reranking can improve precision by reordering the fused candidates.


## Specific Tools And Models

We will be specific about the local stack:

- PDF loader: `LangChain PyPDFLoader`
- Chunker: `LangChain RecursiveCharacterTextSplitter`
- Embedding model: `sentence-transformers/all-MiniLM-L6-v2`
- Vector index: `FAISS`
- Keyword retriever: `rank-bm25 BM25Okapi`
- Reranker: `cross-encoder/ms-marco-MiniLM-L-6-v2`
- Optional generator: `Ollama` with `gemma4:e4b`

Install packages inside the active venv/kernel if needed.


In [1]:
import sys

print("Python executable:", sys.executable)

# Uncomment if your notebook kernel is missing dependencies.
# !{sys.executable} -m pip install -U pypdf langchain-community langchain-text-splitters
# !{sys.executable} -m pip install -U langchain-huggingface sentence-transformers faiss-cpu
# !{sys.executable} -m pip install -U rank-bm25 ollama


Python executable: /Users/jatin.sangwan/dev/RAG_Techniques/venv/bin/python


In [3]:
from pathlib import Path
import re
from typing import Iterable

import numpy as np
import ollama
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder


In [4]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "local" else Path.cwd()
PDF_PATH = PROJECT_ROOT / "data" / "Understanding_Climate_Change.pdf"

GENERATION_MODEL_NAME = "gemma4:e4b"
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

print("Project root:", PROJECT_ROOT)
print("PDF path:", PDF_PATH)
print("PDF exists:", PDF_PATH.exists())
print("Embedding model:", EMBEDDING_MODEL_NAME)
print("Reranker model:", RERANKER_MODEL_NAME)
print("Generation model:", GENERATION_MODEL_NAME)


Project root: /Users/jatin.sangwan/dev/RAG_Techniques
PDF path: /Users/jatin.sangwan/dev/RAG_Techniques/data/Understanding_Climate_Change.pdf
PDF exists: True
Embedding model: sentence-transformers/all-MiniLM-L6-v2
Reranker model: cross-encoder/ms-marco-MiniLM-L-6-v2
Generation model: gemma4:e4b


## Step 1: Tokenization For BM25

BM25 is lexical, so tokenization matters.

For learning, we use a simple tokenizer:

```text
lowercase text
-> keep word and number tokens
```

In production, tokenization may need stemming, stop-word handling, language-specific processing, or the tokenizer built into your search engine.


In [5]:
def tokenize(text: str) -> list[str]:
    """
    Simple BM25 tokenizer.
    """
    return re.findall(r"\b\w+\b", text.lower())


sample_text = "Climate change affects biodiversity, oceans, and weather patterns."
print(tokenize(sample_text))


['climate', 'change', 'affects', 'biodiversity', 'oceans', 'and', 'weather', 'patterns']


## Step 2: Score Normalization

BM25 and vector search return scores on different scales.

Example:

```text
BM25 scores: 0, 3.2, 7.8, 14.1
Vector distances: 0.12, 0.34, 0.51
```

We cannot add these directly. One scale would dominate the other.

So we normalize scores into a `0` to `1` range before fusion.

For FAISS distance-like scores, lower is usually better, so we invert those after normalization.


## Note: How Both Score Types Become `Higher = Better`

Min-max normalization only changes the scale. By itself, it does not know whether a score is good or bad.

The formula is:

```text
normalized = (score - min_score) / (max_score - min_score)
```

This means:

```text
smallest raw value -> 0
largest raw value -> 1
```

For BM25, this is already correct because BM25 means:

```text
higher raw score = better keyword match
```

So for BM25 we use:

```python
higher_is_better=True
return normalized
```

For FAISS distance-like scores, this is backwards because distance usually means:

```text
lower raw distance = better semantic match
```

Example:

```text
FAISS distances: [0.2, 0.5, 0.9]
meaning:          best, medium, worst
```

Normalizing alone gives:

```text
0.2 -> 0
0.5 -> 0.428
0.9 -> 1
```

That would make the worst distance look best. So we invert it:

```python
higher_is_better=False
return 1 - normalized
```

Now:

```text
0.2 -> 1
0.5 -> 0.572
0.9 -> 0
```

After this, both retrievers follow the same rule:

```text
closer to 1 = better chunk
closer to 0 = worse chunk
```

That shared meaning is what makes this fusion formula valid:

```python
fusion_score = alpha * dense_score + (1 - alpha) * bm25_score
```


In [8]:
def min_max_normalize(values: Iterable[float], higher_is_better: bool = True) -> np.ndarray:
    """
    Convert scores to a 0-1 range so BM25 and vector scores can be combined.
    """
    scores = np.array(list(values), dtype=float)
    print(scores)
    score_range = scores.max() - scores.min()

    if score_range == 0:
        return np.ones_like(scores)

    normalized = (scores - scores.min()) / score_range
    print(normalized)

    if higher_is_better:
        return normalized

    return 1 - normalized


print("BM25 normalized:", min_max_normalize([0, 3.2, 7.8, 14.1], higher_is_better=True))
print("Vector distance normalized and inverted:", min_max_normalize([0.12, 0.34, 0.51], higher_is_better=False))


[ 0.   3.2  7.8 14.1]
[0.         0.22695035 0.55319149 1.        ]
BM25 normalized: [0.         0.22695035 0.55319149 1.        ]
[0.12 0.34 0.51]
[0.         0.56410256 1.        ]
Vector distance normalized and inverted: [1.         0.43589744 0.        ]


## Step 3: Load And Chunk The PDF

The document flow is:

```text
PDF
-> page documents
-> text chunks
-> metadata with chunk_id
```

We add a stable `chunk_id` because BM25 scores and vector scores must refer to the same chunk.


In [ ]:
def load_and_chunk_pdf(pdf_path: Path) -> list[Document]:
    loader = PyPDFLoader(str(pdf_path))
    page_documents = loader.load()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        length_function=len,
    )

    chunks = splitter.split_documents(page_documents)

    for chunk_id, chunk in enumerate(chunks):
        chunk.page_content = chunk.page_content.replace("\t", " ")
        chunk.metadata["chunk_id"] = chunk_id

    return chunks


chunks = load_and_chunk_pdf(PDF_PATH)

print("Chunks:", len(chunks))
print("First chunk metadata:", chunks[0].metadata)
print("First chunk preview:")
print(chunks[0].page_content[:500])


## Step 4: Build Dense Vector Search

Dense vector search uses embeddings.

Here we use:

```text
sentence-transformers/all-MiniLM-L6-v2
```

This is a small, fast local embedding model. It is good for learning and local experiments.

The vector index is:

```text
FAISS
```


In [ ]:
def build_dense_index(chunks: list[Document]) -> FAISS:
    embedding_model = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME)
    return FAISS.from_documents(chunks, embedding_model)


vector_store = build_dense_index(chunks)
print("Vector store:", type(vector_store))


## Step 5: Build BM25 Keyword Search

BM25 uses tokenized text chunks.

It does not understand deep semantic meaning, but it is strong for exact words and phrases.


In [ ]:
def build_bm25_index(chunks: list[Document]) -> BM25Okapi:
    tokenized_chunks = [tokenize(chunk.page_content) for chunk in chunks]
    return BM25Okapi(tokenized_chunks)


bm25_index = build_bm25_index(chunks)
print("BM25 index:", type(bm25_index))


## Step 6: Score All Chunks With Both Retrievers

For this small learning PDF, scoring all chunks is fine.

In production, you usually retrieve candidate sets from each retriever instead of scoring every document.
For example:

```text
BM25 top 100
+ vector top 100
-> merge candidates
-> rerank top candidates
```


In [ ]:
def dense_scores_for_all_chunks(
    query: str,
    vector_store: FAISS,
    total_chunks: int,
) -> dict[int, float]:
    dense_results = vector_store.similarity_search_with_score(query, k=total_chunks)

    distances_by_chunk_id = {
        doc.metadata["chunk_id"]: score
        for doc, score in dense_results
    }

    normalized_dense_scores = min_max_normalize(
        distances_by_chunk_id.values(),
        higher_is_better=False,
    )

    return {
        chunk_id: normalized_score
        for chunk_id, normalized_score in zip(
            distances_by_chunk_id.keys(),
            normalized_dense_scores,
        )
    }


def bm25_scores_for_all_chunks(
    query: str,
    bm25_index: BM25Okapi,
    chunks: list[Document],
) -> dict[int, float]:
    raw_bm25_scores = bm25_index.get_scores(tokenize(query))
    normalized_bm25_scores = min_max_normalize(raw_bm25_scores, higher_is_better=True)

    return {
        chunk.metadata["chunk_id"]: score
        for chunk, score in zip(chunks, normalized_bm25_scores)
    }


## Step 7: Fusion Retrieval

Fusion score:

```python
fusion_score = alpha * dense_score + (1 - alpha) * bm25_score
```

Meaning:

- `alpha = 0.8`: mostly semantic/vector search
- `alpha = 0.5`: balanced
- `alpha = 0.2`: mostly keyword/BM25 search


In [ ]:
def fusion_retrieve(
    query: str,
    chunks: list[Document],
    vector_store: FAISS,
    bm25_index: BM25Okapi,
    top_k: int = 5,
    alpha: float = 0.5,
) -> list[tuple[Document, dict[str, float]]]:
    if not 0 <= alpha <= 1:
        raise ValueError("alpha must be between 0 and 1")

    dense_scores = dense_scores_for_all_chunks(query, vector_store, len(chunks))
    bm25_scores = bm25_scores_for_all_chunks(query, bm25_index, chunks)

    scored_results = []
    for chunk in chunks:
        chunk_id = chunk.metadata["chunk_id"]
        dense_score = dense_scores[chunk_id]
        bm25_score = bm25_scores[chunk_id]
        fusion_score = (alpha * dense_score) + ((1 - alpha) * bm25_score)

        scored_results.append(
            (
                chunk,
                {
                    "dense_score": float(dense_score),
                    "bm25_score": float(bm25_score),
                    "fusion_score": float(fusion_score),
                },
            )
        )

    return sorted(
        scored_results,
        key=lambda item: item[1]["fusion_score"],
        reverse=True,
    )[:top_k]


In [ ]:
def print_results(title: str, results: list[tuple[Document, dict[str, float]]]) -> None:
    print(f"\n{title}")
    print("=" * len(title))

    for rank, (doc, scores) in enumerate(results, start=1):
        page = doc.metadata.get("page")
        chunk_id = doc.metadata.get("chunk_id")
        preview = doc.page_content[:350].replace("\n", " ")

        print(f"\nRank {rank} | page={page} | chunk_id={chunk_id}")
        print("Scores:", {key: round(value, 4) for key, value in scores.items()})
        print("Preview:", preview)


## Step 8: Compare Alpha Values

Now we test the same question with different `alpha` values.

Watch how the top chunks change when we lean more toward BM25 or vector search.


In [ ]:
query = "What are the impacts of climate change on biodiversity?"

for alpha in [0.2, 0.5, 0.8]:
    fused_results = fusion_retrieve(
        query=query,
        chunks=chunks,
        vector_store=vector_store,
        bm25_index=bm25_index,
        top_k=5,
        alpha=alpha,
    )
    print_results(f"Fusion retrieval results | alpha={alpha}", fused_results)


## Step 9: Add Reranking After Fusion

This is the production pattern we care about:

```text
hybrid retrieval for recall
-> reranking for precision
```

Fusion gives us a better candidate pool. The reranker then scores each `(query, chunk)` pair directly.


In [ ]:
def rerank_candidates(
    query: str,
    candidate_results: list[tuple[Document, dict[str, float]]],
    top_k: int = 3,
) -> list[tuple[Document, dict[str, float]]]:
    reranker = CrossEncoder(RERANKER_MODEL_NAME)
    pairs = [[query, doc.page_content] for doc, _ in candidate_results]
    reranker_scores = reranker.predict(pairs)

    reranked_results = []
    for (doc, scores), reranker_score in zip(candidate_results, reranker_scores):
        updated_scores = dict(scores)
        updated_scores["reranker_score"] = float(reranker_score)
        reranked_results.append((doc, updated_scores))

    return sorted(
        reranked_results,
        key=lambda item: item[1]["reranker_score"],
        reverse=True,
    )[:top_k]


In [ ]:
fused_candidates = fusion_retrieve(
    query=query,
    chunks=chunks,
    vector_store=vector_store,
    bm25_index=bm25_index,
    top_k=10,
    alpha=0.5,
)

reranked_results = rerank_candidates(query, fused_candidates, top_k=3)
print_results("Hybrid retrieval followed by reranking", reranked_results)


## Step 10: Optional Generation With Ollama

Once we have the final retrieved context, we can send only the best chunks to the local LLM.

This step uses:

```text
Ollama model: gemma4:e4b
```

Make sure Ollama is running and the model is available before executing this cell.


In [ ]:
def answer_with_ollama(query: str, retrieved_docs: list[Document]) -> str:
    context = "\n\n".join(doc.page_content for doc in retrieved_docs)

    prompt = f"""
Answer the question using only the context below.
If the answer is not in the context, say:
"I don't know based on the provided context."

Question:
{query}

Context:
{context}

Answer in 2-4 clear sentences:
"""

    response = ollama.chat(
        model=GENERATION_MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
    )

    return response.message.content


In [ ]:
final_docs = [doc for doc, _ in reranked_results]
answer = answer_with_ollama(query, final_docs)

print(answer)


## Production Takeaways

Use this mental model:

```text
Vector search:
Good for meaning.

BM25:
Good for exact wording.

Hybrid retrieval:
Good for recall.

Reranking:
Good for precision.

Generation:
Only as good as the retrieved context.
```

A common production retrieval architecture is:

```text
query
-> BM25 retrieves candidates
-> vector search retrieves candidates
-> merge/fuse candidates
-> rerank candidates
-> send top context to LLM
-> answer with citations
```

What to evaluate:

- Dense-only retrieval
- BM25-only retrieval
- Fusion retrieval with `alpha = 0.2`, `0.5`, `0.8`
- Fusion retrieval followed by reranking
- Latency and whether the quality gain is worth the cost
